In [ ]:
# ============================================================
# Predicting Airline Passenger Satisfaction Using Machine Learning
# A Data-Driven Approach to Enhancing Customer Experience
# ============================================================


# ============================================================
# STEP 1 : IMPORT LIBRARIES
# Libraries = pre-built tools made by other programmers
# We use them so we don't have to write everything from scratch
# ============================================================

import pandas as pd
# pandas  → helps us work with tables (like Excel in Python)
# pd      → short nickname so we type pd instead of pandas

import numpy as np
# numpy   → helps us do math operations on big lists of numbers
# np      → short nickname

import matplotlib.pyplot as plt
# matplotlib → helps us draw charts and graphs
# pyplot     → the specific part that draws charts
# plt        → short nickname

import seaborn as sns
# seaborn → builds on matplotlib to make prettier, smarter charts
# sns     → short nickname

from sklearn.preprocessing import LabelEncoder, StandardScaler
# sklearn        → a huge library full of Machine Learning tools
# LabelEncoder   → converts text labels (Male/Female) into numbers (0/1)
# StandardScaler → scales all numbers to the same range (needed for Logistic Regression)

from sklearn.model_selection import train_test_split, cross_val_score
# train_test_split → splits our data into training set and testing set
# cross_val_score  → tests the model multiple times to get a reliable accuracy

from sklearn.linear_model import LogisticRegression
# LogisticRegression → our 1st ML model (draws a line to separate classes)

from sklearn.tree import DecisionTreeClassifier
# DecisionTreeClassifier → our 2nd ML model (makes yes/no decisions like a flowchart)

from sklearn.ensemble import RandomForestClassifier
# RandomForestClassifier → our 3rd ML model (combines many decision trees and votes)

from xgboost import XGBClassifier
# XGBClassifier → our 4th ML model (builds trees one by one, fixing mistakes each time)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
# accuracy_score       → calculates what % of predictions were correct
# classification_report → shows precision, recall, F1 score for each class
# confusion_matrix     → table showing correct vs wrong predictions
# ConfusionMatrixDisplay → draws the confusion matrix as a colored chart

print("Libraries imported successfully!")
# print() → shows a message on screen so we know this step worked


# ============================================================
# STEP 2 : LOAD THE DATASET
# We read our CSV file (like opening an Excel file in Python)
# ============================================================

df = pd.read_csv("DS-DATA.csv", low_memory=False)
# pd.read_csv()   → opens the CSV file and loads it into Python
# "DS-DATA.csv"   → name of our data file (must be in the same folder)
# low_memory=False → avoids a warning when column data types are mixed
# df              → short for "dataframe" — it holds our entire table

df["Flight Distance"] = pd.to_numeric(df["Flight Distance"], errors="coerce")
# df["Flight Distance"] → selects the "Flight Distance" column
# pd.to_numeric()       → converts values in that column to proper numbers
# errors="coerce"       → if a value can't be converted, replace it with NaN (empty)
# This fixes a mixed datatype warning that appeared when loading the file

print("uploaded dataset successfully!")
# confirms the file was loaded without errors


# ============================================================
# STEP 3 : EXPLORE / UNDERSTAND THE DATA
# Before doing anything, we look at the data carefully
# This is called Exploratory Data Analysis (EDA)
# ============================================================

print("\nShape:", df.shape)
# df.shape   → tells us (number of rows, number of columns)
# \n         → means "new line" — adds a blank line before the output

print(df.head())
# df.head()  → shows the first 5 rows so we can see what the data looks like

print("\n Info:"); print(df.info())
# df.info()  → shows column names, data types, and how many non-null (non-empty) values
# Helps us understand: which columns are numbers and which are text

print("\n Missing Values:\n", df.isnull().sum())
# df.isnull()    → marks every empty cell as True
# .sum()         → counts how many True values (= missing values) per column
# This tells us which columns have missing data and how many

print("\n Summary:\n", df.describe())
# df.describe() → gives min, max, mean, median for all number columns
# Helps us spot OUTLIERS (very extreme values like delay of 1592 minutes)


# ============================================================
# STEP 4 : TARGET VARIABLE ANALYSIS
# Target = the column we want to predict = "Satisfaction"
# We check: how many passengers are Satisfied vs Not Satisfied?
# ============================================================

print("\n Satisfaction Count:\n", df["Satisfaction"].value_counts())
# df["Satisfaction"]   → selects only the Satisfaction column
# .value_counts()      → counts how many of each unique value (Satisfied / Neutral...)

sns.countplot(x="Satisfaction", data=df, palette="Set2")
# sns.countplot() → draws a bar chart counting each category
# x="Satisfaction" → put Satisfaction values on the x-axis (horizontal)
# data=df          → use our dataframe as the data source
# palette="Set2"   → use a nice color scheme called "Set2"

plt.title("Customer Satisfaction Distribution")
# plt.title() → adds a title to the chart

plt.tight_layout()
# plt.tight_layout() → automatically adjusts spacing so nothing is cut off

plt.show()
# plt.show() → displays the chart on screen


# ============================================================
# STEP 5 : EDA CHARTS — Finding Business Pain Points
# We make charts to visually see which factors affect satisfaction
# This directly addresses the Business Goal
# ============================================================

# ---- Chart 1 : Satisfaction by Travel Class ----
sns.countplot(x="Class", hue="Satisfaction", data=df, palette="Set1")
# x="Class"          → x-axis shows travel class (Economy, Business, Eco Plus)
# hue="Satisfaction" → colors each bar based on satisfaction level
# palette="Set1"     → color scheme

plt.title("Satisfaction by Travel Class")
# adds title to chart

plt.tight_layout(); plt.show()
# cleans up spacing and shows the chart
# INSIGHT: Business class passengers are usually more satisfied


# ---- Chart 2 : Satisfaction by Type of Travel ----
sns.countplot(x="Type of Travel", hue="Satisfaction", data=df, palette="Set2")
# x="Type of Travel" → x-axis shows Business Travel vs Personal Travel

plt.title("Satisfaction by Travel Purpose")
plt.tight_layout(); plt.show()
# INSIGHT: Business travelers tend to be more satisfied


# ---- Chart 3 : Satisfaction by Customer Type ----
sns.countplot(x="Customer Type", hue="Satisfaction", data=df, palette="Set3")
# x="Customer Type" → shows First-time vs Returning customers

plt.title("Satisfaction by Customer Type")
plt.tight_layout(); plt.show()
# INSIGHT: Returning customers tend to be more satisfied


# ---- Chart 4 : Age Distribution by Satisfaction ----
df.groupby("Satisfaction")["Age"].plot(kind="hist", alpha=0.6, bins=30, legend=True)
# df.groupby("Satisfaction") → splits data into groups: Satisfied and Not Satisfied
# ["Age"]                    → look at the Age column within each group
# .plot(kind="hist")         → draw a histogram (bar chart showing frequency)
# alpha=0.6                  → make bars slightly transparent so overlapping bars are visible
# bins=30                    → divide the age range into 30 equal sections
# legend=True                → show a legend explaining which color = which group

plt.title("Age Distribution by Satisfaction")
plt.tight_layout(); plt.show()
# INSIGHT: Middle-aged passengers tend to be more satisfied


# ---- Chart 5 : Box Plots to Detect Outliers in Delay Columns ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# plt.subplots(1, 2) → create 1 row and 2 columns of charts side by side
# figsize=(10, 4)    → width=10 inches, height=4 inches
# fig                → the overall figure (container for both charts)
# axes               → list of 2 individual chart areas [axes[0], axes[1]]

sns.boxplot(y="Departure Delay", data=df, ax=axes[0], color="skyblue")
# sns.boxplot()       → draws a box plot (shows median, spread, and outliers)
# y="Departure Delay" → shows Departure Delay on the y-axis (vertical)
# ax=axes[0]          → place this chart in the LEFT position
# color="skyblue"     → fill the box with sky blue color

sns.boxplot(y="Arrival Delay", data=df, ax=axes[1], color="salmon")
# same as above but for Arrival Delay, placed in the RIGHT position
# Dots far above the box = OUTLIERS (very extreme delay values)

axes[0].set_title("Departure Delay - Outliers?")
# set_title() → adds a title to just this one chart (axes[0])

axes[1].set_title("Arrival Delay - Outliers?")
# adds title to the second chart (axes[1])

plt.tight_layout(); plt.show()


# ---- Service Rating Box Plots: Each Service vs Satisfaction ----
service_cols = [
    "Ease of Online Booking",
    "Online Boarding",
    "Seat Comfort",
    "Cleanliness",
    "Food and Drink",
    "In-flight Wifi Service",
    "In-flight Entertainment"
]
# service_cols → a list of service-related column names we want to analyze

for col in service_cols:
# for loop → repeats the code inside for each column in the list

    plt.figure(figsize=(6, 4))
    # plt.figure() → creates a new chart window
    # figsize=(6,4) → set size: 6 inches wide, 4 inches tall

    sns.boxplot(x="Satisfaction", y=col, data=df)
    # x="Satisfaction" → group by Satisfied (1) vs Not Satisfied (0)
    # y=col            → show the service rating on the y-axis
    # This lets us compare ratings between satisfied and dissatisfied groups

    plt.title(f"{col} vs Satisfaction")
    # f"..." → f-string: inserts the value of col into the title text

    plt.tight_layout()
    plt.show()
    # shows a separate chart for each service column


# ============================================================
# STEP 6 : HANDLE MISSING VALUES
# Some cells in our data are empty (NaN = Not a Number)
# We fill them with the MEDIAN of that column
# Median = middle value — better than mean for skewed data
# ============================================================

for col in df.select_dtypes(include=np.number).columns:
# df.select_dtypes(include=np.number) → selects only numeric columns
# .columns                            → gets just the column names
# for col in ...                      → loop through each numeric column

    df[col] = df[col].fillna(df[col].median())
    # df[col].fillna()    → fills all NaN (empty) values in this column
    # df[col].median()    → with the median (middle value) of that column
    # df[col] =           → saves the updated column back

print("\n Missing values filled with median!")


# ============================================================
# STEP 7 : CAP OUTLIERS IN DELAY COLUMNS
# Departure Delay max was 1592 minutes (over 26 hours!)
# These extreme values mislead the model
# We cap them at the 99th percentile
# (99th percentile = value below which 99% of the data falls)
# ============================================================

for col in ["Departure Delay", "Arrival Delay"]:
# loop through both delay columns

    if col in df.columns:
    # if col in df.columns → check if this column exists in our data (safety check)

        cap = df[col].quantile(0.99)
        # df[col].quantile(0.99) → finds the value at the 99th percentile
        # cap                    → we store this as the maximum allowed value

        df[col] = df[col].clip(upper=cap)
        # df[col].clip(upper=cap) → any value above "cap" gets replaced with "cap"
        # This removes extreme outliers without deleting those rows

print(" Outliers capped at 99th percentile!")


# ============================================================
# STEP 8 : DROP ID COLUMN
# ID is just a unique row number — it has no prediction value
# Keeping it would confuse the model
# ============================================================

if "ID" in df.columns:
# checks if the "ID" column exists

    df.drop("ID", axis=1, inplace=True)
    # df.drop()    → removes a column from the dataframe
    # "ID"         → name of the column to remove
    # axis=1       → axis=1 means column (axis=0 means row)
    # inplace=True → make the change directly on df (don't create a copy)

    print(" ID column dropped!")


# ============================================================
# STEP 9 : ENCODE CATEGORICAL COLUMNS (Text → Numbers)
# ML models only understand numbers
# Columns like "Gender" have text values (Male/Female)
# LabelEncoder converts: Female→0, Male→1
# ============================================================

label_encoders = {}
# label_encoders → empty dictionary to store encoders for each column
# We save them in case we need to reverse the encoding later

for col in df.select_dtypes(include="object").columns:
# select_dtypes(include="object") → selects only text columns
# loop through each text column

    if col != "Satisfaction":
    # skip the Satisfaction column — we handle it separately in Step 10

        le = LabelEncoder()
        # le → create a new LabelEncoder object for this column

        df[col] = le.fit_transform(df[col])
        # le.fit_transform() → two steps in one:
        #   fit      → learns all unique values in the column
        #   transform → converts those values to numbers (0, 1, 2...)

        label_encoders[col] = le
        # save this encoder in our dictionary so we can use it later if needed

print(" Categorical columns encoded!")


# ============================================================
# STEP 10 : ENCODE TARGET COLUMN
# We encode "Satisfaction" separately with its own encoder
# so we can easily reverse it later (numbers back to text)
# ============================================================

target_encoder = LabelEncoder()
# create a separate LabelEncoder just for the target column

df["Satisfaction"] = target_encoder.fit_transform(df["Satisfaction"])
# converts Satisfaction column:
#   "Neutral or Dissatisfied" → 0
#   "Satisfied"               → 1

print("\n Target Encoding:", dict(zip(target_encoder.classes_,
                                      target_encoder.transform(target_encoder.classes_))))
# target_encoder.classes_      → list of original text labels
# target_encoder.transform(...) → list of numbers they map to
# dict(zip(...))               → combines both into a readable dictionary
# Example output: {'Neutral or Dissatisfied': 0, 'Satisfied': 1}


# ============================================================
# STEP 11 : CLASS IMBALANCE CHECK
# If 90% = Satisfied and 10% = Not Satisfied,
# the model can just predict "Satisfied" always and get 90% accuracy
# without actually learning anything — that's BAD
# We check if the split is too unequal
# ============================================================

class_counts = df["Satisfaction"].value_counts()
# counts how many rows belong to each class (0 and 1)

min_pct = class_counts.min() / len(df) * 100
# class_counts.min() → count of the smaller class
# len(df)            → total number of rows
# * 100             → convert to percentage
# min_pct           → percentage of the minority class

use_class_weight = min_pct < 35
# if minority class is less than 35% → data is imbalanced
# use_class_weight = True if imbalanced, False if balanced

if use_class_weight:
    print("  Imbalanced data! Applying class_weight='balanced'")
else:
    print(" Data is balanced!")
# prints which situation we're in

scale_pos_weight = (
    class_counts.max() / class_counts.min()
    if use_class_weight
    else 1
)
# scale_pos_weight → used by XGBoost to handle imbalanced data
# if imbalanced: ratio of majority to minority class
# if balanced: just 1 (no adjustment needed)


# ============================================================
# STEP 12 : CORRELATION HEATMAP
# Correlation = how strongly two columns relate to each other
# Value of 1  → perfectly related (both go up together)
# Value of -1 → oppositely related (one up, other down)
# Value of 0  → no relationship
# ============================================================

plt.figure(figsize=(16, 12))
# creates a large figure: 16 wide, 12 tall

sns.heatmap(
    df.select_dtypes(include=np.number).corr(),
    # df.select_dtypes() → use only numeric columns
    # .corr()            → calculate correlation between every pair of columns
    # Result = a table where each cell shows correlation between 2 columns

    cmap="coolwarm",
    # coolwarm → color scheme: blue=negative, white=zero, red=positive

    linewidths=0.3,
    # adds thin lines between cells so they're easier to read

    vmin=-1, vmax=1
    # vmin/vmax → set color scale range from -1 to 1
)

plt.title("Correlation Heatmap")
plt.tight_layout(); plt.show()


# ============================================================
# STEP 13 & 14 : SPLIT DATA INTO FEATURES AND TARGET, THEN TRAIN/TEST
# X = Input features (what we feed to the model)
# y = Target (what we want the model to predict)
# We split: 80% training + 20% testing
# ============================================================

X = df.drop("Satisfaction", axis=1)
# df.drop("Satisfaction", axis=1) → remove Satisfaction column
# X = all remaining columns = the features (inputs)

y = df["Satisfaction"]
# y = only the Satisfaction column = the target (output we want to predict)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    # the features and target we want to split

    test_size=0.20,
    # 20% of data goes to testing, 80% to training

    random_state=42,
    # random_state → seed for randomness — ensures same split every time we run

    stratify=y
    # stratify=y → makes sure both train and test sets have the same % of each class
    # Without this, by chance one set might have way more "Satisfied" than the other
)

print(f"\n Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
# shape[0] → number of rows
# f"..." → f-string inserts variable values into the text


# ============================================================
# STEP 15 : FEATURE SCALING
# Logistic Regression gets confused when features have very different ranges
# Example: "Age" ranges 7-85 but "Flight Distance" ranges 31-4983
# StandardScaler brings all features to same scale (mean=0, std=1)
# IMPORTANT: We learn the scale from TRAINING data only
# Then apply the SAME scale to test data — this prevents data leakage
# Data leakage = test data accidentally influencing the model during training
# ============================================================

scaler = StandardScaler()
# creates a StandardScaler object

X_train_scaled = scaler.fit_transform(X_train)
# fit_transform = two steps:
#   fit       → learns mean and std from training data
#   transform → scales training data using those values

X_test_scaled = scaler.transform(X_test)
# transform only → applies the SAME mean and std learned from training
# We do NOT fit again on test data — that would be data leakage

print("Scaling done!")


# ============================================================
# STEP 16 : TRAIN 4 ML MODELS AND COMPARE
# We train 4 different models and see which performs best
# ============================================================

cw = "balanced" if use_class_weight else None
# cw = "balanced" → tells LR and RF to adjust for imbalanced classes
# cw = None       → no adjustment (when data is already balanced)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight=cw, random_state=42),
    # max_iter=1000  → allow up to 1000 iterations to find the best line
    # class_weight   → handles imbalanced data if needed
    # random_state   → ensures reproducible results

    "Decision Tree"      : DecisionTreeClassifier(random_state=42),
    # A simple flowchart-like model that asks yes/no questions

    "Random Forest"      : RandomForestClassifier(n_estimators=100, class_weight=cw, random_state=42),
    # n_estimators=100 → build 100 decision trees and take a majority vote

    "XGBoost"            : XGBClassifier(eval_metric="logloss", random_state=42, verbosity=0, scale_pos_weight=scale_pos_weight)
    # eval_metric="logloss"   → how XGBoost measures its own performance during training
    # verbosity=0             → don't print training messages (keeps output clean)
    # scale_pos_weight        → handles class imbalance for XGBoost specifically
}
# models → a dictionary: key = model name, value = model object

results        = []
# empty list to store accuracy results for each model

best_accuracy  = 0
# starts at 0 — any model will beat this

best_model     = None
# will store the actual best model object

best_model_name = ""
# will store the name of the best model as a string

print("\n Training Models...\n" + "-"*45)

for name, model in models.items():
# models.items() → gives us both the name and the model object each loop
# name  = "Logistic Regression", "Decision Tree", etc.
# model = the actual model object

    Xtr = X_train_scaled if name == "Logistic Regression" else X_train
    # Logistic Regression needs scaled data (StandardScaler applied)
    # Tree models work fine with original unscaled data

    Xte = X_test_scaled if name == "Logistic Regression" else X_test
    # same logic for test data

    model.fit(Xtr, y_train)
    # model.fit() → TRAINS the model
    # Xtr         → the input features for training
    # y_train     → the correct answers for training
    # The model learns patterns from this

    y_pred = model.predict(Xte)
    # model.predict() → uses the trained model to PREDICT on test data
    # Xte             → test features (model has never seen these)
    # y_pred          → predicted labels (0 or 1) for each test row

    accuracy = accuracy_score(y_test, y_pred)
    # accuracy_score() → compares predicted labels vs actual labels
    # y_test           → the real correct answers
    # y_pred           → what our model predicted
    # Returns a value like 0.9661 (= 96.61%)

    cv = cross_val_score(model, Xtr, y_train, cv=5, scoring="accuracy")
    # cross_val_score → runs 5-fold cross validation
    # cv=5            → splits training data into 5 parts
    #                   trains on 4 parts, tests on 1 part, 5 times total
    # This gives a more RELIABLE accuracy than a single train-test split
    # cv is now an array of 5 accuracy scores

    results.append([name, round(accuracy*100,2), round(cv.mean()*100,2), round(cv.std()*100,2)])
    # .append()       → adds a new row to our results list
    # round(..., 2)   → round to 2 decimal places
    # cv.mean()       → average of the 5 cross-val scores
    # cv.std()        → how much the 5 scores vary (lower = more stable)

    print(f"  {name}")
    print(f"    Test Accuracy : {round(accuracy*100,2)}%")
    print(f"    CV Accuracy   : {round(cv.mean()*100,2)}% ± {round(cv.std()*100,2)}%\n")
    # ± means "plus or minus" — shows how stable the model is

    if accuracy > best_accuracy:
    # if this model is better than the current best

        best_accuracy, best_model, best_model_name = accuracy, model, name
        # update all three best-model variables at once

print(f" Best Model : {best_model_name}  ({round(best_accuracy*100,2)}%)")


# ============================================================
# STEP 17 : MODEL COMPARISON BAR CHART
# Visualise all 4 model accuracies side by side
# ============================================================

comp_df = pd.DataFrame(results, columns=["Model", "Test Acc %", "CV Mean %", "CV Std %"])
# pd.DataFrame()  → converts our results list into a proper table
# columns=[...]   → names for each column in the table

comp_df = comp_df.sort_values("Test Acc %", ascending=False)
# sort_values()   → sort the table by accuracy (highest first)
# ascending=False → highest values come first

colors = ["#2ecc71" if n == best_model_name else "#95a5a6" for n in comp_df["Model"]]
# list comprehension → creates a color for each model
# #2ecc71 = green  → best model gets green bar
# #95a5a6 = gray   → all other models get gray bar

bars = plt.bar(comp_df["Model"], comp_df["Test Acc %"], color=colors)
# plt.bar()  → draws a vertical bar chart
# x = model names, y = accuracy values, color = our color list

plt.title("Model Accuracy Comparison  (Green = Best)")
plt.ylim(70, 100)
# plt.ylim() → y-axis starts at 70 (not 0) so differences look more visible

plt.xticks(rotation=15)
# plt.xticks(rotation=15) → rotate x-axis labels by 15 degrees so they don't overlap

for b, v in zip(bars, comp_df["Test Acc %"]):
# zip(bars, values) → pairs each bar with its accuracy value
# b = a single bar, v = its accuracy number

    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2, f"{v}%", ha="center", fontsize=10)
    # plt.text()              → adds text on the chart
    # b.get_x()               → left edge of the bar
    # + b.get_width()/2       → adds half the bar width → center of bar
    # b.get_height() + 0.2    → just above the top of the bar
    # f"{v}%"                 → the text to show (e.g. "96.61%")
    # ha="center"             → horizontally centered

plt.tight_layout(); plt.show()


# ============================================================
# STEP 18 & 19 : EVALUATE BEST MODEL IN DETAIL
# Classification Report → Precision, Recall, F1
# Confusion Matrix → shows correct vs wrong predictions visually
# ============================================================

Xte_best = X_test_scaled if best_model_name == "Logistic Regression" else X_test
# use scaled test data for LR, original test data for all others

y_pred_best = best_model.predict(Xte_best)
# use the best model to predict on test data

print("\n Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=target_encoder.classes_))
# classification_report() → shows for each class:
#   Precision → of all predicted Satisfied, how many were truly Satisfied?
#   Recall    → of all truly Satisfied, how many did we correctly predict?
#   F1-Score  → balance between Precision and Recall (higher = better)
#   Support   → how many actual examples of that class in test set
# target_names → show class names as text instead of 0 and 1

cm = confusion_matrix(y_test, y_pred_best)
# confusion_matrix() → creates a 2x2 table:
#   Row 0, Col 0 → True Negative  (predicted NOT, actually NOT)   ✅
#   Row 0, Col 1 → False Positive (predicted Satisfied, actually NOT) ❌
#   Row 1, Col 0 → False Negative (predicted NOT, actually Satisfied) ❌
#   Row 1, Col 1 → True Positive  (predicted Satisfied, actually Satisfied) ✅

ConfusionMatrixDisplay(cm, display_labels=target_encoder.classes_).plot(cmap="Blues", colorbar=False)
# ConfusionMatrixDisplay → turns the confusion matrix table into a colored chart
# display_labels         → show class names not 0/1
# cmap="Blues"           → use blue color scale
# colorbar=False         → don't show the color legend bar on the side
# .plot()                → actually draw and display the chart

plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout(); plt.show()


# ============================================================
# STEP 20 : FEATURE IMPORTANCE
# Which columns/features mattered MOST for predictions?
# This is the KEY business insight of the project
# Higher importance = more influence on whether a passenger is satisfied
# ============================================================

print("\n Top 10 Features Driving Satisfaction:")

if best_model_name != "Logistic Regression":
# Tree-based models (Decision Tree, RF, XGBoost) have .feature_importances_

    imp_df = pd.DataFrame({"Feature": X.columns, "Importance": best_model.feature_importances_})
    # X.columns                    → list of all feature names
    # best_model.feature_importances_ → array of importance scores (one per feature)
    # pd.DataFrame()               → combine them into a table

    imp_df = imp_df.sort_values("Importance", ascending=False).head(10)
    # sort by importance (highest first) and keep only top 10

    colors = sns.color_palette("RdYlGn", 10)[::-1]
    # "RdYlGn" → color palette: Red (low) → Yellow (medium) → Green (high)
    # [::-1]   → reverses the list so green = most important (at top)

    plt.barh(imp_df["Feature"][::-1], imp_df["Importance"][::-1], color=colors)
    # plt.barh()   → horizontal bar chart (features on y-axis)
    # [::-1]       → reverse order so most important feature is at the TOP

else:
# For Logistic Regression: use the coefficient values instead
# Larger absolute coefficient = more influence on prediction

    imp_df = pd.DataFrame({"Feature": X.columns, "Importance": np.abs(best_model.coef_[0])})
    # best_model.coef_[0] → array of coefficients for each feature
    # np.abs()            → take absolute value (ignore + or - sign, just size)

    imp_df = imp_df.sort_values("Importance", ascending=False).head(10)

    plt.barh(imp_df["Feature"][::-1], imp_df["Importance"][::-1], color="steelblue")
    # steelblue → a single solid color for Logistic Regression bars

plt.xlabel("Importance Score")
# label for the x-axis (horizontal axis)

plt.title(f"Top 10 Features — {best_model_name}")
plt.tight_layout(); plt.show()

print(imp_df.to_string(index=False))
# .to_string(index=False) → prints the table without the row numbers on the left


# ---- Pain Point Analysis ----
print("\n PAIN POINT ANALYSIS")
# This shows the AVERAGE rating for each service
# separated by satisfied vs dissatisfied passengers

rating_cols = [
    "Ease of Online Booking",
    "Online Boarding",
    "Seat Comfort",
    "Cleanliness",
    "Food and Drink",
    "In-flight Wifi Service",
    "In-flight Entertainment"
]
# list of service rating columns we want to analyze

pain_points = (
    df.groupby("Satisfaction")[rating_cols]
    # df.groupby("Satisfaction") → split data into group 0 and group 1
    # [rating_cols]              → look at only our service rating columns

      .mean()
      # .mean() → calculate the average rating for each service within each group

      .round(2)
      # .round(2) → round to 2 decimal places
)

print(pain_points)
# Shows: for each service, what's the avg rating for Satisfied vs Dissatisfied
# A big gap between the two rows = that service is a KEY PAIN POINT


# ============================================================
# STEP 21 : BUSINESS INSIGHTS
# Summary of findings with real numbers from our data
# These are the actionable recommendations for the airline
# ============================================================

print("""
              BUSINESS INSIGHTS
==================================================
TOP FACTORS AFFECTING SATISFACTION (From Our Data):
  1. Online Boarding (43.6%)   → Improve the boarding app experience
  2. Type of Travel  (12.8%)   → Business travelers need priority service
  3. In-flight Wifi  (10.1%)   → Upgrade in-flight connectivity
  4. Customer Type   (6.6%)    → Loyalty rewards for returning customers
  5. In-flight Entertainment (4.4%) → Better content & streaming options
""")
# These numbers come directly from our Step 20 Feature Importance output
# This section connects the ML results to real business actions


# ============================================================
# STEP 22, 23, 24 : PREDICT ALL PASSENGERS + SEGMENT THEM
# Now we predict satisfaction for ALL 129,880 passengers
# Then assign each one to a business segment
# ============================================================

Xa = scaler.transform(X) if best_model_name == "Logistic Regression" else X
# if best model is LR: scale all features (using the scaler already fitted)
# otherwise: use original unscaled features
# X here = ALL rows (not just train or test)

all_preds = target_encoder.inverse_transform(best_model.predict(Xa))
# best_model.predict(Xa)            → predict 0 or 1 for every passenger
# target_encoder.inverse_transform() → convert 0 back to "Neutral or Dissatisfied"
#                                      and 1 back to "Satisfied"

all_probas = best_model.predict_proba(Xa)[:, 1] * 100
# best_model.predict_proba() → gives probability of being each class
#   Output shape: (129880, 2) — two columns: prob of class 0, prob of class 1
# [:, 1]                     → take only the second column (probability of being Satisfied)
# * 100                      → convert from 0.9999 to 99.99%

def segment(prob):
# defines a function that takes a probability and returns a segment name

    if prob >= 80:
        return "Highly Satisfied"
        # prob ≥ 80% → very likely satisfied → keep them happy

    elif prob >= 50:
        return "Neutral"
        # 50% ≤ prob < 80% → on the fence → could go either way

    else:
        return "Highly Dissatisfied(at risk)"
        # prob < 50% → likely dissatisfied → needs immediate attention

output_df = pd.read_csv("DS-DATA.csv", low_memory=False)
# reload the original dataset (with original text columns, not encoded numbers)
# We add predictions to this original version so output is readable

output_df["Predicted_Satisfaction"] = all_preds
# add a new column with predicted label for each passenger

output_df["Satisfaction_Probability%"] = all_probas.round(2)
# add a new column with the probability % of being Satisfied

output_df["Customer_Segment"] = [segment(p) for p in all_probas]
# list comprehension → calls segment() for every probability value
# adds a new column with the segment label (Highly Satisfied / Neutral / At Risk)

print(" Sample Output:")
print(output_df[["Predicted_Satisfaction", "Satisfaction_Probability%", "Customer_Segment"]].head(5))
# show first 5 rows of the 3 new columns we just added


# ============================================================
# STEP 25 : SEGMENTATION SUMMARY CHART
# Bar chart showing how many passengers fall in each segment
# ============================================================

seg_colors = {"Highly Satisfied": "#2ecc71", "Neutral": "#f39c12", "Highly Dissatisfied": "#e74c3c"}
# dictionary mapping segment names to colors
# green = happy, orange = neutral, red = at risk

seg_counts = output_df["Customer_Segment"].value_counts()
# counts how many passengers are in each segment

seg_counts.plot(kind="bar", color=[seg_colors.get(s, "gray") for s in seg_counts.index])
# .plot(kind="bar")         → draw a bar chart
# seg_colors.get(s, "gray") → look up color for each segment; use gray if not found

plt.title("Customer Segments")
plt.xticks(rotation=15)
# rotate labels so they don't overlap

plt.tight_layout(); plt.show()


# ============================================================
# STEP 26 : SAVE OUTPUT FILES
# Save our results to CSV files so the airline can use them
# ============================================================

output_df.to_csv("Airline_Satisfaction_Output.csv", index=False)
# .to_csv()    → saves the dataframe as a CSV file
# index=False  → don't add a row number column to the file

output_df.to_csv("Final_Airline_Customer_Report.csv", index=False)
# saves the same data again with a different filename (the business report version)

print("\n Output saved:  Airline_Satisfaction_Output.csv")
print(" Report saved:  Final_Airline_Customer_Report.csv")


# ============================================================
# STEP 27 : FINAL SUMMARY
# Prints a clean summary of everything we accomplished
# ============================================================

print(f"""
{'='*50}
 PROJECT COMPLETE
{'='*50}
 Best Model    : {best_model_name}
 Test Accuracy : {round(best_accuracy*100,2)}%
""")
# f"""..."""    → multi-line f-string
# {'='*50}     → prints "=" 50 times as a divider line
# {best_model_name}          → inserts the name of the best model
# {round(best_accuracy*100,2)} → inserts the best accuracy as a percentage


# ============================================================
# BONUS : TOP 5 AT-RISK CUSTOMERS
# Shows the passengers most likely to be dissatisfied
# The airline should contact these people first!
# ============================================================

print("\nTop 5 At-Risk Customers")
print(
    output_df[
        ["ID", "Satisfaction_Probability%", "Customer_Segment"]
        # select only these 3 columns
    ]
    .sort_values(by="Satisfaction_Probability%")
    # sort by probability in ASCENDING order (lowest probability = most at risk at top)

    .head()
    # .head() → show only the top 5 rows (the 5 most at-risk passengers)
)
# These are passengers with 0% probability of being satisfied
# The airline can reach out to them proactively to improve their experience

In [ ]:
# ============================================================
# Predicting Airline Passenger Satisfaction Using Machine Learning
# ============================================================


# ============================================================
# STEP 1 : IMPORT LIBRARIES
# ============================================================

import pandas as pd          # work with data tables
import numpy as np           # math on arrays
import matplotlib.pyplot as plt  # draw charts
import seaborn as sns        # prettier charts

from sklearn.preprocessing import LabelEncoder, StandardScaler
# LabelEncoder → text to numbers | StandardScaler → normalize ranges

from sklearn.model_selection import train_test_split, cross_val_score
# train_test_split → split data | cross_val_score → reliable accuracy check

from sklearn.linear_model import LogisticRegression   # model 1
from sklearn.tree import DecisionTreeClassifier        # model 2
from sklearn.ensemble import RandomForestClassifier    # model 3
from xgboost import XGBClassifier                      # model 4

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
# accuracy_score → % correct | classification_report → precision/recall/F1 | confusion_matrix → correct vs wrong

print("Libraries imported successfully!")


# ============================================================
# STEP 2 : LOAD THE DATASET
# ============================================================

df = pd.read_csv("DS-DATA.csv", low_memory=False)
# load CSV into a table; low_memory=False avoids mixed-type warning

df["Flight Distance"] = pd.to_numeric(df["Flight Distance"], errors="coerce")
# convert Flight Distance to number; non-convertible → NaN

print("uploaded dataset successfully!")


# ============================================================
# STEP 3 : EXPLORE / UNDERSTAND THE DATA (EDA)
# ============================================================

print("\nShape:", df.shape)          # (rows, columns)
print(df.head())                     # first 5 rows
print("\n Info:"); print(df.info())  # column names, types, non-null counts
print("\n Missing Values:\n", df.isnull().sum())  # missing count per column
print("\n Summary:\n", df.describe())             # min, max, mean per column


# ============================================================
# STEP 4 : TARGET VARIABLE ANALYSIS
# ============================================================

print("\n Satisfaction Count:\n", df["Satisfaction"].value_counts())
# count of Satisfied vs Not Satisfied

sns.countplot(x="Satisfaction", data=df, palette="Set2")
# bar chart of satisfaction split
plt.title("Customer Satisfaction Distribution")
plt.tight_layout()
plt.show()


# ============================================================
# STEP 5 : EDA CHARTS — Finding Business Pain Points
# ============================================================

# ---- Chart 1 : Satisfaction by Travel Class ----
sns.countplot(x="Class", hue="Satisfaction", data=df, palette="Set1")
# insight: Business class → most satisfied
plt.title("Satisfaction by Travel Class")
plt.tight_layout(); plt.show()

# ---- Chart 2 : Satisfaction by Type of Travel ----
sns.countplot(x="Type of Travel", hue="Satisfaction", data=df, palette="Set2")
# insight: Business travelers → more satisfied
plt.title("Satisfaction by Travel Purpose")
plt.tight_layout(); plt.show()

# ---- Chart 3 : Satisfaction by Customer Type ----
sns.countplot(x="Customer Type", hue="Satisfaction", data=df, palette="Set3")
# insight: Returning customers → more satisfied
plt.title("Satisfaction by Customer Type")
plt.tight_layout(); plt.show()

# ---- Chart 4 : Age Distribution by Satisfaction ----
df.groupby("Satisfaction")["Age"].plot(kind="hist", alpha=0.6, bins=30, legend=True)
# insight: Middle-aged passengers → more satisfied
plt.title("Age Distribution by Satisfaction")
plt.tight_layout(); plt.show()

# ---- Chart 5 : Box Plots — Delay Outliers ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # 2 charts side by side
sns.boxplot(y="Departure Delay", data=df, ax=axes[0], color="skyblue")
sns.boxplot(y="Arrival Delay",   data=df, ax=axes[1], color="salmon")
# dots above box = extreme outliers
axes[0].set_title("Departure Delay - Outliers?")
axes[1].set_title("Arrival Delay - Outliers?")
plt.tight_layout(); plt.show()

# ---- Service Rating Box Plots ----
service_cols = [
    "Ease of Online Booking", "Online Boarding", "Seat Comfort",
    "Cleanliness", "Food and Drink", "In-flight Wifi Service", "In-flight Entertainment"
]

for col in service_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x="Satisfaction", y=col, data=df)
    # compares each service rating: satisfied vs dissatisfied
    plt.title(f"{col} vs Satisfaction")
    plt.tight_layout()
    plt.show()


# ============================================================
# STEP 6 : HANDLE MISSING VALUES
# ============================================================

for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())
    # fill empty cells with column median (robust to skewed data)

print("\n Missing values filled with median!")


# ============================================================
# STEP 7 : CAP OUTLIERS IN DELAY COLUMNS
# ============================================================

for col in ["Departure Delay", "Arrival Delay"]:
    if col in df.columns:
        cap = df[col].quantile(0.99)       # 99th percentile = max allowed value
        df[col] = df[col].clip(upper=cap)  # replace anything above cap with cap

print(" Outliers capped at 99th percentile!")


# ============================================================
# STEP 8 : DROP ID COLUMN
# ============================================================

if "ID" in df.columns:
    df.drop("ID", axis=1, inplace=True)  # ID has no predictive value → remove
    print(" ID column dropped!")


# ============================================================
# STEP 9 : ENCODE CATEGORICAL COLUMNS (Text → Numbers)
# ============================================================

label_encoders = {}  # store encoders in case we need to reverse later

for col in df.select_dtypes(include="object").columns:
    if col != "Satisfaction":             # handle target separately in Step 10
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])  # learn values + convert to numbers
        label_encoders[col] = le

print(" Categorical columns encoded!")


# ============================================================
# STEP 10 : ENCODE TARGET COLUMN
# ============================================================

target_encoder = LabelEncoder()
df["Satisfaction"] = target_encoder.fit_transform(df["Satisfaction"])
# Neutral or Dissatisfied → 0 | Satisfied → 1

print("\n Target Encoding:", dict(zip(target_encoder.classes_,
                                      target_encoder.transform(target_encoder.classes_))))


# ============================================================
# STEP 11 : CLASS IMBALANCE CHECK
# ============================================================

class_counts    = df["Satisfaction"].value_counts()
min_pct         = class_counts.min() / len(df) * 100  # minority class %
use_class_weight = min_pct < 35  # True = imbalanced; apply correction

if use_class_weight:
    print("  Imbalanced data! Applying class_weight='balanced'")
else:
    print(" Data is balanced!")

scale_pos_weight = (class_counts.max() / class_counts.min()
                    if use_class_weight else 1)
# ratio used by XGBoost to correct for imbalance


# ============================================================
# STEP 12 : CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(16, 12))
sns.heatmap(
    df.select_dtypes(include=np.number).corr(),  # correlation table of all numeric columns
    cmap="coolwarm",   # blue = negative, red = positive
    linewidths=0.3,
    vmin=-1, vmax=1
)
plt.title("Correlation Heatmap")
plt.tight_layout(); plt.show()


# ============================================================
# STEP 13 & 14 : SPLIT DATA — FEATURES / TARGET + TRAIN/TEST
# ============================================================

X = df.drop("Satisfaction", axis=1)  # all columns except target = features
y = df["Satisfaction"]               # target column to predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,    # 80% train, 20% test
    random_state=42,   # fixed seed → same split every run
    stratify=y         # keep same class ratio in both sets
)

print(f"\n Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")


# ============================================================
# STEP 15 : FEATURE SCALING
# ============================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # learn scale from train, apply
X_test_scaled  = scaler.transform(X_test)        # apply SAME scale to test (no leakage)

print("Scaling done!")


# ============================================================
# STEP 16 : TRAIN 4 ML MODELS AND COMPARE
# ============================================================

cw = "balanced" if use_class_weight else None  # class weight setting for LR and RF

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight=cw, random_state=42),
    # draws a decision boundary; needs scaled data
    "Decision Tree"      : DecisionTreeClassifier(random_state=42),
    # yes/no flowchart
    "Random Forest"      : RandomForestClassifier(n_estimators=100, class_weight=cw, random_state=42),
    # 100 trees vote together
    "XGBoost"            : XGBClassifier(eval_metric="logloss", random_state=42, verbosity=0,
                                         scale_pos_weight=scale_pos_weight)
    # builds trees sequentially, each fixing previous errors
}

results         = []
best_accuracy   = 0
best_model      = None
best_model_name = ""

print("\n Training Models...\n" + "-"*45)

for name, model in models.items():
    Xtr = X_train_scaled if name == "Logistic Regression" else X_train  # LR needs scaled data
    Xte = X_test_scaled  if name == "Logistic Regression" else X_test

    model.fit(Xtr, y_train)       # train
    y_pred   = model.predict(Xte) # predict on unseen test data
    accuracy = accuracy_score(y_test, y_pred)  # % correct

    cv = cross_val_score(model, Xtr, y_train, cv=5, scoring="accuracy")
    # 5-fold CV → split train into 5, rotate through for stable accuracy

    results.append([name, round(accuracy*100,2), round(cv.mean()*100,2), round(cv.std()*100,2)])

    print(f"  {name}")
    print(f"    Test Accuracy : {round(accuracy*100,2)}%")
    print(f"    CV Accuracy   : {round(cv.mean()*100,2)}% ± {round(cv.std()*100,2)}%\n")

    if accuracy > best_accuracy:
        best_accuracy, best_model, best_model_name = accuracy, model, name

print(f" Best Model : {best_model_name}  ({round(best_accuracy*100,2)}%)")


# ============================================================
# STEP 17 : MODEL COMPARISON BAR CHART
# ============================================================

comp_df = pd.DataFrame(results, columns=["Model", "Test Acc %", "CV Mean %", "CV Std %"])
comp_df = comp_df.sort_values("Test Acc %", ascending=False)  # highest first

colors = ["#2ecc71" if n == best_model_name else "#95a5a6" for n in comp_df["Model"]]
# green = best model, gray = others

bars = plt.bar(comp_df["Model"], comp_df["Test Acc %"], color=colors)
plt.title("Model Accuracy Comparison  (Green = Best)")
plt.ylim(70, 100)       # start y-axis at 70 so differences are visible
plt.xticks(rotation=15)

for b, v in zip(bars, comp_df["Test Acc %"]):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2, f"{v}%", ha="center", fontsize=10)
    # label each bar with its accuracy value

plt.tight_layout(); plt.show()


# ============================================================
# STEP 18 & 19 : EVALUATE BEST MODEL IN DETAIL
# ============================================================

Xte_best     = X_test_scaled if best_model_name == "Logistic Regression" else X_test
y_pred_best  = best_model.predict(Xte_best)

print("\n Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=target_encoder.classes_))
# Precision = how many predicted positives were right
# Recall    = how many actual positives were caught
# F1        = balance of precision & recall

cm = confusion_matrix(y_test, y_pred_best)
# [TN, FP]
# [FN, TP]  → correct predictions on diagonal

ConfusionMatrixDisplay(cm, display_labels=target_encoder.classes_).plot(cmap="Blues", colorbar=False)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout(); plt.show()


# ============================================================
# STEP 20 : FEATURE IMPORTANCE
# ============================================================

print("\n Top 10 Features Driving Satisfaction:")

if best_model_name != "Logistic Regression":
    imp_df = pd.DataFrame({"Feature": X.columns, "Importance": best_model.feature_importances_})
    # feature_importances_ → how much each column contributed to predictions

    imp_df = imp_df.sort_values("Importance", ascending=False).head(10)

    colors = sns.color_palette("RdYlGn", 10)[::-1]  # green = most important
    plt.barh(imp_df["Feature"][::-1], imp_df["Importance"][::-1], color=colors)

else:
    imp_df = pd.DataFrame({"Feature": X.columns, "Importance": np.abs(best_model.coef_[0])})
    # coef_ absolute value → feature influence for Logistic Regression

    imp_df = imp_df.sort_values("Importance", ascending=False).head(10)
    plt.barh(imp_df["Feature"][::-1], imp_df["Importance"][::-1], color="steelblue")

plt.xlabel("Importance Score")
plt.title(f"Top 10 Features — {best_model_name}")
plt.tight_layout(); plt.show()
print(imp_df.to_string(index=False))

# ---- Pain Point Analysis ----
print("\n PAIN POINT ANALYSIS")

rating_cols = [
    "Ease of Online Booking", "Online Boarding", "Seat Comfort",
    "Cleanliness", "Food and Drink", "In-flight Wifi Service", "In-flight Entertainment"
]

pain_points = (
    df.groupby("Satisfaction")[rating_cols]
      .mean()
      .round(2)
)
# big gap between satisfied (1) and dissatisfied (0) = key pain point
print(pain_points)


# ============================================================
# STEP 21 : BUSINESS INSIGHTS
# ============================================================

print("""
              BUSINESS INSIGHTS
==================================================
TOP FACTORS AFFECTING SATISFACTION (From Our Data):
  1. Online Boarding (43.6%)        → Improve boarding app experience
  2. Type of Travel  (12.8%)        → Business travelers need priority service
  3. In-flight Wifi  (10.1%)        → Upgrade in-flight connectivity
  4. Customer Type   (6.6%)         → Loyalty rewards for returning customers
  5. In-flight Entertainment (4.4%) → Better content & streaming options
""")


# ============================================================
# STEP 22, 23, 24 : PREDICT ALL PASSENGERS + SEGMENT THEM
# ============================================================

Xa = scaler.transform(X) if best_model_name == "Logistic Regression" else X
# scale all rows if LR, otherwise use as-is

all_preds  = target_encoder.inverse_transform(best_model.predict(Xa))
# predict for all 129,880 passengers → convert 0/1 back to text labels

all_probas = best_model.predict_proba(Xa)[:, 1] * 100
# probability (%) of being Satisfied for each passenger

def segment(prob):
    if   prob >= 80: return "Highly Satisfied"
    elif prob >= 50: return "Neutral"
    else:            return "Highly Dissatisfied(at risk)"
# bucket each passenger based on their satisfaction probability

output_df = pd.read_csv("DS-DATA.csv", low_memory=False)  # reload original (readable) data
output_df["Predicted_Satisfaction"]  = all_preds
output_df["Satisfaction_Probability%"] = all_probas.round(2)
output_df["Customer_Segment"] = [segment(p) for p in all_probas]

print(" Sample Output:")
print(output_df[["Predicted_Satisfaction", "Satisfaction_Probability%", "Customer_Segment"]].head(5))


# ============================================================
# STEP 25 : SEGMENTATION SUMMARY CHART
# ============================================================

seg_colors  = {"Highly Satisfied": "#2ecc71", "Neutral": "#f39c12", "Highly Dissatisfied": "#e74c3c"}
seg_counts  = output_df["Customer_Segment"].value_counts()

seg_counts.plot(kind="bar", color=[seg_colors.get(s, "gray") for s in seg_counts.index])
# green = happy | orange = neutral | red = at risk
plt.title("Customer Segments")
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()


# ============================================================
# STEP 26 : SAVE OUTPUT FILES
# ============================================================

output_df.to_csv("Airline_Satisfaction_Output.csv",   index=False)
output_df.to_csv("Final_Airline_Customer_Report.csv", index=False)
# index=False → don't save row numbers

print("\n Output saved:  Airline_Satisfaction_Output.csv")
print(" Report saved:  Final_Airline_Customer_Report.csv")


# ============================================================
# STEP 27 : FINAL SUMMARY
# ============================================================

print(f"""
{'='*50}
 PROJECT COMPLETE
{'='*50}
 Best Model    : {best_model_name}
 Test Accuracy : {round(best_accuracy*100,2)}%
""")


# ============================================================
# BONUS : TOP 5 AT-RISK CUSTOMERS
# ============================================================

print("\nTop 5 At-Risk Customers")
print(
    output_df[["ID", "Satisfaction_Probability%", "Customer_Segment"]]
    .sort_values(by="Satisfaction_Probability%")  # lowest probability first = most at risk
    .head()
)
# airline should proactively reach out to these passengers